## Notebook script to create a pMHC single-chain trimer
Create a tool that is able to create the nucleotide sequence to insert into desired destination vector. With this tool, you should be able to insert a peptide sequence and select the HLA allele and it will generate the nucleotide sequence you would order for Gibson

In [3]:
import sys
sys.path.insert(0, "..")
sys.path.insert(0, "../tools")

import pandas as pd
from config import ALL_DATA_DIR

In [30]:
# dictionary for linkers
linker_nuc = {
    'L1' : "ggtggaggaggttctggaggtggtggtagtggtggtggtggttcc",
    'L2' : "ggtggtggtggtagtggtggtggtggttcaggtggtggtggttccggtggtggtggttcc"
}

# dictionary for B2M
B2M_nuc = "atacaaagaactccaaagatccaagtttacagtagacatcctgctgaaaacggtaaatctaatttcttgaactgttacgtctccggtttccacccaagtgatatagaagttgacttgttgaaaaatggtgaaagaatcgaaaaggttgaacattcagatttgtctttttctaaggactggtccttctatttgttgtactacacagaattcactccaactgaaaaggatgaatacgcttgcagagttaatcatgtaaccttgtctcaacctaaaatcgttaagtgggatagagacatg"

# dictionary for destination vectors
destination_vector_nuc = {
    "pHIV-EGFP" : {
        "5_addon": "tcgtgagcggccgctgagtt",
        "3_addon": "aactattctagagtacccgggctaggatcc"},
    "pHIV-dTomato": {
        "5_addon": "gaactgaattcatcgacgtt",
        "3_addon": "aactattctagagtacccgggctaggatcc"}
}

# leader signal peptide
leader_peptide_aa = {
    "B2M": "MSRSVALAVLALLSLSGLEA",
    "HGH": "MATGSRTSLLLAFGLLCLPWLQEGSA"
}

HLA_nuc = {
    "HLA-A2": "catagtatgagatatttctttacttctgtttcaagaccaggtagaggtgaacctagattcatcgcagtcggttacgttgatgacacacaatttgtaagattcgattccgacgctgcaagtcaaagaatggaaccaagagcaccttggattgaacaagaaggtccagaatattgggatggtgaaactagaaaagttaaggcccattctcaaactcacagagtagatttgggtacattaagaggtgcttataatcaatctgaagcaggttcacatacagtacaaagaatgtacggttgtgatgtcggttcagactggagatttttgagaggttatcaccaatatgcttacgatggtaaagactacattgcattgaaggaagatttgagatcctggaccgccgctgacatggcagcccaaactacaaaacataagtgggaagctgcacacgtagcagaacaattgagagcctatttggaaggtacatgtgtcgaatggttgagaagatacttagaaaacggtaaagaaacattgcaaagaaccgatgctccaaagactcatatgacacatcacgccgttagtgatcacgaagctactttgagatgctgggcattatctttttaccctgccgaaatcacattgacctggcaaagagatggtgaagaccaaacccaagatactgaattagttgaaaccagaccagcaggtgacggtactttccaaaaatgggccgctgttgtagtcccttcaggtcaagaacaaagatacacatgccatgtccaacacgaaggtttaccaaagccattgacattgagatgggaaccatcc"
}

peptide_nuc = {
    "MART-1": "gaggccgcggggataggtatattgacggta",
    "MART-2": "mart2",
    "MART-3": "mart3",
}

EnforceGCContent:
- makes GC constraint local within the 50 nucleotide window such that each window will fall between 30-70% GC content

In [ ]:
import numpy as np
from numpy import random

def convert_nucleotide(aa_seq, seed=None):
    if seed is None:
        seed = random.randint(0,10000)

In [10]:
import numpy as np
from dnachisel import (DnaOptimizationProblem, CodonOptimize,
                       EnforceTranslation, AvoidPattern, EnforceGCContent)
from dnachisel.biotools import reverse_translate, translate


def convert_nucleotide(aa_seq, seed, species="h_sapiens",
                       avoid_patterns=None,
                       gc_mini=0.2, gc_maxi=0.4, gc_window=50):
    """Codon-optimise one or several amino acid sequences into nucleotides.

    dnachisel draws all of its randomness from numpy's global RNG (both the
    randomised back-translation and the optimisation itself), so seeding that
    RNG is what makes a run reproducible. The seed is therefore required: no
    sequence is generated without one, so every sequence on record can be
    regenerated by calling this again with the same seed and the same inputs.

    Parameters
    ----------
    aa_seq
        A single amino acid sequence, a list of them, or a dict mapping a
        name to a sequence (e.g. leader_peptide_aa).
    seed
        Required int, used to seed numpy's global RNG once for the whole
        call. In a batch the sequences are generated from one continuing
        random stream, so reproducing any of them means rerunning the same
        call, not just the one sequence.
    species
        Species for the CodonOptimize objective.
    avoid_patterns
        Optional patterns to forbid. Accepts a single pattern ("BsaI_site")
        or a list of them. If None, no patterns are avoided.
    gc_mini, gc_maxi, gc_window
        Local GC content constraint, applied within each sliding window.

    Returns
    -------
    dna
        The optimised nucleotide sequence(s), matching the shape of aa_seq:
        a string for a single sequence, a list for a list, a dict for a dict.
    """
    if seed is None:
        raise ValueError(
            "a seed is required: the sequence cannot be reproduced without "
            "one, so pass an explicit seed (e.g. seed=1234) before running. "
            "Draw one with int(np.random.randint(0, 1_000_000)) and record it."
        )
    np.random.seed(int(seed))

    if avoid_patterns is None:
        avoid_patterns = []
    elif isinstance(avoid_patterns, str):
        avoid_patterns = [avoid_patterns]

    # Normalise the input to a (keys, sequences) pair, remembering its shape
    # so the results can be handed back in the same form.
    if isinstance(aa_seq, str):
        keys, sequences, shape = [None], [aa_seq], "single"
    elif isinstance(aa_seq, dict):
        keys, sequences, shape = list(aa_seq), list(aa_seq.values()), "dict"
    else:
        sequences = list(aa_seq)
        keys, shape = list(range(len(sequences))), "list"
    sequences = ["".join(s.split()).upper() for s in sequences]

    results = []
    for protein in sequences:
        problem = DnaOptimizationProblem(
            sequence=reverse_translate(protein, randomize_codons=True),
            constraints=[EnforceTranslation(),
                         *[AvoidPattern(pattern) for pattern in avoid_patterns],
                         EnforceGCContent(mini=gc_mini, maxi=gc_maxi,
                                          window=gc_window)],
            objectives=[CodonOptimize(species=species)],
            logger=None,
        )
        problem.resolve_constraints()
        problem.optimize()

        dna = problem.sequence
        assert translate(dna) == protein, \
            "translation does not match input protein"
        results.append(dna)

    if shape == "single":
        return results[0]
    if shape == "dict":
        return dict(zip(keys, results))
    return results


In [13]:
convert_nucleotide(
    leader_peptide_aa,
    seed=123)

{'B2M': 'ATGAGCAGATCTGTTGCTTTAGCTGTTCTTGCCTTATTATCTTTATCTGGACTGGAAGCT',
 'HGH': 'ATGGCTACAGGCTCTAGGACATCTCTTCTTTTAGCTTTTGGATTATTATGTCTTCCTTGGTTGCAAGAAGGAAGCGCC'}

Created function is able to output nucleotide sequence

### Creating nucleotide sequence

Section will then create a standard nucleotide sequence for whatever aa input.
If input is a dictionary, will iterate and replace the value with 

Chain joins as:
SP-Peptide-L1-B2M-L2-HLA

In [26]:
seed = 1231

SP = convert_nucleotide(
    leader_peptide_aa,
    seed=seed)

In [28]:
seq = [SP['B2M'], peptide_nuc['MART-1'], linker_nuc["L1"], linker_nuc["L2"], HLA_nuc['HLA-A2']]
to_order = "".join(seq)
to_order

'ATGTCTAGATCTGTTGCTTTGGCTGTTTTAGCTCTTCTTTCTTTATCTGGACTTGAAGCTgaggccgcggggataggtatattgacggtaggtggaggaggttctggaggtggtggtagtggtggtggtggttccggtggtggtggtagtggtggtggtggttcaggtggtggtggttccggtggtggtggttcccatagtatgagatatttctttacttctgtttcaagaccaggtagaggtgaacctagattcatcgcagtcggttacgttgatgacacacaatttgtaagattcgattccgacgctgcaagtcaaagaatggaaccaagagcaccttggattgaacaagaaggtccagaatattgggatggtgaaactagaaaagttaaggcccattctcaaactcacagagtagatttgggtacattaagaggtgcttataatcaatctgaagcaggttcacatacagtacaaagaatgtacggttgtgatgtcggttcagactggagatttttgagaggttatcaccaatatgcttacgatggtaaagactacattgcattgaaggaagatttgagatcctggaccgccgctgacatggcagcccaaactacaaaacataagtgggaagctgcacacgtagcagaacaattgagagcctatttggaaggtacatgtgtcgaatggttgagaagatacttagaaaacggtaaagaaacattgcaaagaaccgatgctccaaagactcatatgacacatcacgccgttagtgatcacgaagctactttgagatgctgggcattatctttttaccctgccgaaatcacattgacctggcaaagagatggtgaagaccaaacccaagatactgaattagttgaaaccagaccagcaggtgacggtactttccaaaaatgggccgctgttgtagtcccttcaggtcaagaacaaagatacacatgccatgtccaacacgaaggtttaccaaagccattg

In [31]:
to_order_all = {
    name: "".join([SP['B2M'], peptide, linker_nuc["L1"], linker_nuc["L2"], HLA_nuc['HLA-A2']])
    for name, peptide in peptide_nuc.items()
}
to_order_all

{'MART-1': 'ATGTCTAGATCTGTTGCTTTGGCTGTTTTAGCTCTTCTTTCTTTATCTGGACTTGAAGCTgaggccgcggggataggtatattgacggtaggtggaggaggttctggaggtggtggtagtggtggtggtggttccggtggtggtggtagtggtggtggtggttcaggtggtggtggttccggtggtggtggttcccatagtatgagatatttctttacttctgtttcaagaccaggtagaggtgaacctagattcatcgcagtcggttacgttgatgacacacaatttgtaagattcgattccgacgctgcaagtcaaagaatggaaccaagagcaccttggattgaacaagaaggtccagaatattgggatggtgaaactagaaaagttaaggcccattctcaaactcacagagtagatttgggtacattaagaggtgcttataatcaatctgaagcaggttcacatacagtacaaagaatgtacggttgtgatgtcggttcagactggagatttttgagaggttatcaccaatatgcttacgatggtaaagactacattgcattgaaggaagatttgagatcctggaccgccgctgacatggcagcccaaactacaaaacataagtgggaagctgcacacgtagcagaacaattgagagcctatttggaaggtacatgtgtcgaatggttgagaagatacttagaaaacggtaaagaaacattgcaaagaaccgatgctccaaagactcatatgacacatcacgccgttagtgatcacgaagctactttgagatgctgggcattatctttttaccctgccgaaatcacattgacctggcaaagagatggtgaagaccaaacccaagatactgaattagttgaaaccagaccagcaggtgacggtactttccaaaaatgggccgctgttgtagtcccttcaggtcaagaacaaagatacacatgccatgtccaacacgaaggtttac

### Including API from IEDB

Identified that single chain trimers cannot use the classical HLAs found. Mutations at certain points are required. The heavy chain has to have been validated to be expressed on cell surface. 

In [32]:
import requests

BASE = "https://www.ebi.ac.uk/cgi-bin/ipd/api"

# single allele by accession
r = requests.get(f"{BASE}/allele/HLA00001", params={"project": "HLA"})
data = r.json()
data["name"]      # 'A*01:01:01:01'
data["locus"]     # 'A*'

'A*'

In [33]:
r = requests.get(f"{BASE}/allele/download", params={
    "project": "HLA",
    "query": 'startsWith(name,"A*02:01")',
    "type": "coding",
})

In [36]:
import requests
from io import StringIO
from Bio import SeqIO

BASE = "https://www.ebi.ac.uk/cgi-bin/ipd/api"

r = requests.get(f"{BASE}/allele/download", params={
    "project": "HLA",
    "query": 'eq(name,"A*02:01")',
    "type": "coding",        # or 'genomic' or 'protein'
})
r.raise_for_status()

records = list(SeqIO.parse(StringIO(r.text), "fasta"))
str(records[0].seq)

IndexError: list index out of range